In [9]:
from typing import Union

# numpy for working with matrices, etc.
import numpy as np

# import pandas
import pandas as pd

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.ticker import PercentFormatter


# scipy for statistics and optimization
from scipy import optimize
from scipy import stats

# import cvxpy

# function to perform interpolation
from scipy.interpolate import interp1d

"""
functions from codelib
"""

import pandas as pd
# functions for calculating moments
import codelib.Johan.statistics.moments as mom
# functions for calculating risk metrics
import codelib.Johan.portfolio_optimization.risk_metrics as rm
# functions for risk budgetting
import codelib.Johan.portfolio_optimization.risk_budget as rb
# functions for mean-variance optimization
import codelib.Johan.portfolio_optimization.mean_variance as mvo

# cash flows
from codelib.Johan.fixed_income.cash_flows import CashFlow

# predefined plots
from codelib.Johan.visualization.base import fan_chart

In [8]:
"""
Initialize parameters for simulation
"""

np.random.seed(222)

# equity parameters
sigma = 0.15
mu = 0.06

# short rate parameters
initial_rate = 0.03
kappa = 1.0
theta = 0.03
beta = 0.02
rp = -0.2

# correlation
rho = -0.2

# simulation definition
num_sim = 10_000
dt = 1.0 / 12.0
horizon = 5.0
num_time_steps = int(horizon / dt)
time_points = np.arange(0, num_time_steps + 1, 1) * dt

In [15]:
import pandas as pd

# List of files (without .csv extension)
files = [
    "EUNK_iShares_Core_MSCI_Europe_UCITS_ETF_EUR_Acc",
    "IUSN_iShares_MSCI_World_Small_Cap_UCITS_ETF_USD_Acc",
    "JGHY_JPM_Global_High_Yield_Corporate_Bond_Multi-Factor_UCITS_ETF_USD_acc",
    "SXR4_iShares_MSCI_USA_UCITS_ETF_USD_Acc",
    "SYBB_SPDR_Bloomberg_Euro_Government_Bond_UCITS_ETF_Dist",
    "SYBC_SPDR_Bloomberg_Euro_Corporate_Bond_UCITS_ETF_Dist"
]

data_dict = {}
for f in files:
    csv_path = f"../Simulation/Data/{f}.csv"  # Adjust path if needed
    df = pd.read_csv(csv_path)
    df.rename(columns={"Dato": "Date", "Slutkurs": "ClosedPrice"}, inplace=True)
    data_dict[f] = df

combined_df = None
for name, df in data_dict.items():
    short_name = name.split("_")[0]  # e.g. "EUNK", "IUSN", etc.
    temp_df = df[["Date", "ClosedPrice"]].copy()
    temp_df.rename(columns={"ClosedPrice": f"{short_name}_ClosedPrice"}, inplace=True)

    # Use how="inner" so only dates present in *all* CSVs remain
    if combined_df is None:
        combined_df = temp_df
    else:
        combined_df = pd.merge(combined_df, temp_df, on="Date", how="inner")

print(combined_df.head())


         Date  EUNK_ClosedPrice  IUSN_ClosedPrice  JGHY_ClosedPrice  \
0  11/02/2020             57.91             5.058             91.69   
1  12/02/2020             58.29             5.086             91.99   
2  13/02/2020             58.28             5.115             92.36   
3  14/02/2020             58.18             5.123             92.50   
4  24/02/2020             55.60             4.897             91.77   

   SXR4_ClosedPrice  SYBB_ClosedPrice  SYBC_ClosedPrice  
0             297.9             67.08             59.78  
1             299.8             67.14             59.81  
2             301.4             67.33             59.81  
3             301.7             67.22             59.84  
4             289.2             67.56             59.81  


In [16]:
"""
Define functions for simulation
"""

def simulate_vasicek(initial_short_rate: float, kappa: float, theta: float, beta: float, horizon: float,
                     dt: float=1.0/12, num_sim: int=10000, z_mat=None):
    """
    simulates short rate processes in a vasicek setting until a given horizon

    Parameters
    ----------

    initial_short_rate:
        initial short rate
    kappa:
        speed of mean reversion.
    theta:
        long term mean of the short rate.
    dt:
        increments in time
    horizon:
        time until maturity/expiry (horizon).
    num_sim:
        number of simulations.
    """
    std_rates = np.sqrt(beta**2 / (2 * kappa) * (1 - np.exp(-2 * kappa * dt)))

    num_periods = int(horizon / dt)
    short_rates = np.empty((num_sim, num_periods + 1))
    short_rates[:, 0] = initial_short_rate

    if z_mat is None:
        error_terms = np.random.normal(scale=std_rates, size=(num_sim, num_periods))
    else:
        error_terms = std_rates * z_mat

    for i in range(1, num_periods + 1):

        short_rates[:, i] = theta + (short_rates[:, i - 1] - theta) * np.exp(-kappa * dt) + error_terms[:, i - 1]

    return short_rates


def simulate_risk_drivers(mu: float, sigma: float,
                          initial_rate: float, kappa: float, theta: float, beta: float,
                          rho: float,
                          horizon: float,
                          dt: float=1.0 / 12,
                          num_sim: int=10_000):

    """
    Function simulating the risky asset and the short rate.
    """

    # define the number of time steps
    num_time_steps = int(horizon / dt)

    # convert parameters of equity values
    mu_scaled = (mu - 0.5 * sigma**2) * dt
    sigma_scaled = sigma * np.sqrt(dt)

    # define innovation correlation matrix
    z_corr_mat = np.array([[1.0, rho], [rho, 1.0]])

    # simulate innovations
    z_mat = np.random.multivariate_normal(np.zeros(2), z_corr_mat, size=(num_sim, num_time_steps))

    # simulate equity prices
    log_ret = mu_scaled + sigma_scaled * z_mat[:, :, 0]

    equity_prices = np.ones((num_sim, num_time_steps + 1))
    equity_prices[:, 1:] = np.exp(np.cumsum(log_ret, axis=1))

    # simulate short rates
    short_rates = simulate_vasicek(initial_short_rate=initial_rate,
                                   kappa=kappa,
                                   theta=theta,
                                   beta=beta,
                                   horizon=horizon,
                                   dt=dt,
                                   num_sim=num_sim,
                                   z_mat=z_mat[:, :, 1])

    return equity_prices, short_rates

